# Postprocessing

In [ ]:
# Suppress Possible Warnings : In some installations a warning due to pyg_lib path can arise: not relevant to our setting
import warnings
warnings.filterwarnings(
    "ignore",
    message=".*An issue occurred while importing 'pyg-lib'.*",
    category=UserWarning,
    module="torch_geometric.typing"
)
warnings.filterwarnings(
    "ignore",
    message=".*An issue occurred while importing 'torch-sparse'.*",
    category=UserWarning,
    module="torch_geometric.typing"
)

In [ ]:
import re
import torch
import logging
import numpy as np
import networkx as nx
import torch.nn as nn
import pandas as pd
from scipy.stats import iqr
import torch_geometric.utils as pyg_utils
from torch_geometric.utils import to_dense_adj
from torch_geometric import seed_everything
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

from src.config import set_run_dir
from src.config import set_out_dir
from src.config import load_cfg
from src.logger import setup_printing
from src.utils.device import auto_select_device
from src.model.factory import create_model
from src.train.optim import create_optimizer, create_scheduler
from src.dataset.loader import load_data_and_create_dataloader
from src.train.create_trainer import create_trainer
from src.train.optim import create_criterion
from src.utils.plots import plot_node_forecasting
from src.train.task import create_task
from src.metric.anomaly_score import get_detection_metrics
from src.utils.plots import plot_confusion_matrix, plot_roc_auc


# Load trained model

In [ ]:
# to ensure that we all work on the same model, we use the pretrained model
cfg_file = 'run/configs/swat/gdn-pretrained.yaml'
cfg = load_cfg(cfg_file)
set_out_dir(cfg.out_dir, cfg_file)
set_run_dir(cfg.out_dir, cfg.seed)

cfg.print = 'stdout'
cfg.train.shuffle = False
cfg.train.batch_size = 256
auto_select_device()

seed_everything(cfg.seed)
setup_printing(logging.INFO)

In [ ]:
loaders, scaler = load_data_and_create_dataloader()

In [ ]:
model = create_model(cfg.model.type)

In [ ]:
optimizer = create_optimizer(model.parameters())
scheduler = create_scheduler(optimizer)

crit_type = cfg.optim.criterion
criterion = create_criterion(crit_type, mask_loss=False)

task = create_task(
    cfg.task.type,
    task_train_type=cfg.task.train_type,
    track_graph=False,
)

trainer = create_trainer(
    model,
    optimizer=optimizer,
    neptune_writer=None,
    scheduler=scheduler,
    criterion=criterion,
    scaler=scaler,
    task=task,
)

In [ ]:
trainer.load(model)
model.eval()

In [ ]:
train_loader, val_loader, test_loader = loaders

In [ ]:
# read out features from run/datasets/swat/features.txt with column name feature
feature_map = pd.read_csv('run/datasets/swat/features.txt', header=None, names=['feature'])
feature_map = feature_map['feature'].to_dict()
feature_list = list(feature_map.values())
feature_list

# Evaluate the model on the validation and test set

In [ ]:
trainer.eval_epoch(test_loader, 'eval')
eval_preds = torch.cat(trainer.loggers['eval']._pred).detach().cpu().numpy()
eval_trues = torch.cat(trainer.loggers['eval']._true).detach().cpu().numpy()

In [ ]:
trainer.eval_epoch(test_loader, 'test')
test_preds = torch.cat(trainer.loggers['test']._pred).detach().cpu().numpy()
test_trues = torch.cat(trainer.loggers['test']._true).detach().cpu().numpy()
test_true_labels = torch.cat(trainer.loggers['test']._custom_stats['label']).detach().cpu().numpy()

# Evaluate forecasting performance

In [ ]:
f = plot_node_forecasting(eval_preds[:, 0], eval_trues[:, 0], (0, 1000), labels=feature_list)
plt.tight_layout()

In [ ]:
f = plot_node_forecasting(test_preds[:, 0], test_trues[:, 0], (0, 1000), labels=feature_list)
plt.tight_layout()

# Ex1: Anomaly detection performance evaluation

**Objective:**
Calculate the anomaly score and evaluate the anomaly detection performance of a sensor network.

**Tasks:**

1. **Error calculation:**
   - Compute the error for each sensor \(i\) at time \(t\) using:
      $Err_i (t) = |s(t)^{(i)} − \hat{s}^{(t)}|$

2. **Robust sensor normalization:**
   - Normalize the error values to account for different sensor characteristics:
     $a_i^{(t)} =  \frac{Err_i^{(t)} − \tilde{\mu_i}}{\tilde{\sigma_i}}$
   - Here, $\tilde{\mu_i}$ is the median and  $\tilde{\sigma_i}$ is the inter-quartile range (IQR) of $Err_i{(t)}$ over time.

3. **Anomaly score calculation:**
   - Compute the overall anomaly score at time \(t\) by taking the maximum normalized error across all sensors:
     $A(t) = \max_{i} a_i(t)$

4. **Performance evaluation:**
   - Evaluate the anomaly detection performance (F1 score, confusion matric, AUC) using the computed anomaly scores.


## Calucalte the anomaly score and evaluate the anomaly detection performance

In [ ]:
def get_err_median_and_iqr(predicted, groundtruth, rng=(25, 75)):

    np_arr = np.abs(np.subtract(np.array(predicted), np.array(groundtruth)))
    err_median = np.median(np_arr, axis=0)
    err_iqr = iqr(np_arr, rng=rng, axis=0)

    return err_median, err_iqr

def get_ad_score_by_node(pred, true, epsilon=1e-2, rng=(25, 75), n_err_mid=None, n_err_iqr=None):
    pred = pred.squeeze()
    true = true.squeeze()

    if n_err_mid is None and n_err_iqr is None:
        n_err_mid, n_err_iqr = get_err_median_and_iqr(pred, true, rng)

    test_delta = np.abs(np.subtract(
                        np.array(pred).astype(np.float64),
                        np.array(true).astype(np.float64)
                    ))
    # TODO: Define the error_score here
    err_scores = ...

    # smoothing
    smoothed_err_scores = err_scores.copy()
    before_num = 3
    for i in range(before_num, len(err_scores)):
        smoothed_err_scores[i] = np.mean(err_scores[i-before_num:i+1], axis=0)

    return smoothed_err_scores, n_err_mid, n_err_iqr

def get_ad_topk_ad_score(score_by_node, topk=1):
    # select top k errors for each graph (GDN)
    n_nodes = cfg.dataset.n_nodes
    topk_indices = np.argpartition(score_by_node, range(n_nodes-topk-1, n_nodes), axis=1)[:, -topk:]
    score = np.sum(np.take_along_axis(score_by_node, topk_indices, axis=1), axis=1)
    return score


In [ ]:
n_nodes = cfg.dataset.n_nodes
window_size = cfg.dataset.horizon

eval_preds_by_nodes = eval_preds.reshape(-1, n_nodes, window_size)
eval_trues_by_nodes = eval_trues.reshape(-1, n_nodes, window_size)
test_preds_by_nodes = test_preds.reshape(-1, n_nodes, window_size)
test_trues_by_nodes = test_trues.reshape(-1, n_nodes, window_size)

In [ ]:
# Validation score
val_score, err_iqr, err_median = get_ad_score_by_node(
    eval_trues_by_nodes, eval_preds_by_nodes, n_err_iqr=None, n_err_mid=None,
)

# # Test score
test_score, _, _ = get_ad_score_by_node(
    test_trues_by_nodes, test_preds_by_nodes, n_err_iqr=err_iqr, n_err_mid=err_median,
)

In [ ]:
eval_topk_score = get_ad_topk_ad_score(val_score)
test_topk_score = get_ad_topk_ad_score(test_score)

threshold = np.max(eval_topk_score)
pred_labels = test_topk_score > threshold

eval_topk_score.shape
out = get_detection_metrics(test_true_labels, pred_labels, test_topk_score, fault_labels=None, threshold=threshold)
out

## Plot the confusion matrix and ROC curve

In [ ]:
# TODO: plot_confusion_matrix takes tn, fp, fn and tp
fig = plot_confusion_matrix(...,
                            labels=['normal', 'anomaly'],
                            title='Confusion matrix')

In [ ]:
# TODO: plot_roc_auc takes fpr, tpr, and auc as input
fig = plot_roc_auc(...)

# Ex2: Interpret sensor embedding
To explain the learned model, we can visualize its sensor embedding vectors.
Sensors can be grouped by their type (e.g., "FIT", "PIT") or by their component number, which is indicated by the first digit in the sensor name (e.g., "2" in "AIT203").

**Task:** Visualize the sensor embedding vectors in 2D space using t-SNE, grouping the sensors by their type and component number to explore patterns and similarities.


![SWaT Process](img/swat_process.png)

In [ ]:
def extract_sensor_type(component_id):
    return re.match(r'[^\d]+', component_id).group(0)

sensor_component_id = [fn[-3] for fn in feature_list]
sensor_types = [extract_sensor_type(feat) for feat in feature_list]

In [ ]:
sensor_embeddings = model.embedding.weight.detach().cpu().numpy()
sensor_embeddings.shape

tsne = TSNE(n_components=2, verbose=1, perplexity=40, n_iter=300)
tsne_results = tsne.fit_transform(sensor_embeddings)

In [ ]:
# TODO:
df_sensor_embedding = pd.DataFrame({
    'sensor_name': feature_list,
    'sensor_type': sensor_types,
    'sensor_component_id': sensor_component_id,
    'tsne1': ...,
    'tsne2': ...,
})
df_sensor_embedding.head()

In [ ]:
# TODO:
plt.figure(figsize=(5, 5))
sns.scatterplot(
    x='tsne1', y='tsne2',
    style=...,
    hue=...,
    data=df_sensor_embedding,
    legend="full",
    alpha=0.9
)

plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

### Ex3: Interpret learned graph for anomaly localization
Leverage the learned graph from a Graph Neural Network (GNN) to identify and localize anomalies in a sensor network.

**Task:**
1. **Graph Analysis:**
   - Analyze edges to understand relationships between sensors.
   - Use attention weights to determine the importance of neighboring sensors.

2. **Anomaly Localization:**
   - Identify sensors most affected by anomalies.
   - Compare observed sensor readings with expected behavior to explain deviations.

## Show alarm distribution

In [ ]:
plt.figure(figsize=(8, 2))
plt.plot(test_true_labels)
plt.title('True alarm labels')

# TODO: show alarm rates
print('Alarm rate: {:.3f}'.format(... / len(test_true_labels)))

In [ ]:
test_batch = next(iter(test_loader)).to(cfg.device)
g = pyg_utils.to_networkx(test_batch[0])

# get initial adjacency matrix
adj_init = nx.adjacency_matrix(g).toarray()
adj_init

## Visualize the learned graph

In [ ]:
def compute_graph(model: nn.Module, batch):

    batch = batch.to(cfg.device)

    # Do a forward pass in the model
    with torch.no_grad():
        model(batch, None)

    coeff_weights = model.gnn_layers[0].att_weight_1.detach()
    edge_index = model.gnn_layers[0].edge_index_1.detach()
    A = to_dense_adj(edge_index, edge_attr=coeff_weights, batch=batch.batch).squeeze().cpu().numpy()
    return A

In [ ]:
adj_test = compute_graph(model, test_batch)
adj_test.shape

The following code visualizes edges with highest attention score between the provided central node (abnormal node under attack) and the most affected nodes

In [ ]:
def plot_central_node(adj, central_node_id, edge_weight_threshold=0.1, topk=None):
    # fix random seed for reproducibility
    np.random.seed(980452)

    anomaly_node_size = 80
    default_node_size = 20

    central_node_color = "yellow"
    anomaly_node_color = "red"
    default_node_color = "black"

    anomaly_edge_color = "red"
    default_edge_color = (0.35686275, 0.20392157, 0.34901961, 0.1)

    feature_num = cfg.dataset.n_nodes

    central_node = int(central_node_id)

    # Find the neighboring nodes and selected the edges with highest value
    scores = np.stack([adj[central_node], adj[:, central_node]], axis=1)
    scores = np.max(scores, axis=1)

    if topk is None:
        # Define red nodes as the nodes with edge weight > edge_weight_threshold
        red_nodes = list(np.where(scores > edge_weight_threshold)[0])
    else:
        red_nodes = list(np.argsort(scores)[::-1][:topk])

    G = nx.from_numpy_array(adj)
    G.remove_edges_from(nx.selfloop_edges(G))

    edges = [set(edge) for edge in G.edges()]
    edge_colors = [default_edge_color for edge in edges]

    node_colors = [default_node_color for i in range(feature_num)]
    node_sizes = [default_node_size for i in range(feature_num)]

    node_colors[central_node] = central_node_color
    node_sizes[central_node] = anomaly_node_size

    for node in red_nodes:
        if node == central_node:
            continue

        node_colors[node] = anomaly_node_color
        node_sizes[node] = anomaly_node_size

        edge_pos = edges.index(set((node, central_node)))
        edge_colors[edge_pos] = anomaly_edge_color

    pos = nx.spring_layout(G)

    x, y = pos[central_node]
    plt.text(x-0.05,y + 0.1,
                s=feature_map[central_node],
                bbox=dict(facecolor=central_node_color, alpha=0.5), horizontalalignment='center')

    print("Central Node:", feature_map[central_node])

    for node in red_nodes:
        x, y = pos[node]
        plt.text(x-0.05,y + 0.1,
                    s=feature_map[node],
                    bbox=dict(facecolor=anomaly_node_color, alpha=0.5), horizontalalignment='center')

        print("Red Node:", feature_map[node])

    nx.draw(G, pos,
            edge_color=edge_colors,
            node_color=node_colors,
            node_size=node_sizes)


In [ ]:
# for visualization purpose, you can choose any indes to visualize
time_index = ...
central_node = test_score[time_index].argmax()
print(f"Central Node: {central_node}: {feature_map[central_node]}")

plot_central_node(adj_test[time_index], central_node, topk=...)

## Explain attack scenario

In [ ]:
adj_matrices = []
for batch in test_loader:
    batch = batch.to(cfg.device)
    adj = compute_graph(model, batch)
    adj_matrices.append(adj)

adj_matrices = np.concatenate(adj_matrices, axis=0)

## Inspect the following alarms that you have explored in the file gnn_ex2.1_swat_data_expl.ipynb

- LIT301: 2015-12-29T11:57:25 - 2015-12-29T12:02:00 = 275 seconds
- MV101: 2015-12-29T18:30:00 - 2015-12-29T18:42:00 = 720 seconds

![SWaT Process](img/swat_process.png)

In [ ]:
# TODO: you need to find out the time index of the alarms from gnn_ex2.1_swat_data_expl.ipynb
# attck on LIT301
time_index = ...
topk = ... # you can play with the number of affected nodes shown
plot_central_node(adj_matrices[time_index], central_node_id=central_node, topk=topk)

In [ ]:
# plot the forecasting performance around time_index
f = plot_node_forecasting(test_preds[:, 0], test_trues[:, 0], idx_range=(time_index-100, time_index+100), labels=list(feature_map.values()))
plt.tight_layout()

In [ ]:
# TODO: find the time index of attack on MV101
time_index = ...
topk = ...
plot_central_node(adj_matrices[time_index], central_node_id=central_node, topk=)